# Homework 3 Notebook 
Author: Eugenia Tate 

Course: AIPI 530 by Dr. Natalia Summerville 

## Problem 1 — Campus courier tour (TSP)

A courier starts at office **O**, visits locations **A, B, and C** exactly once, and returns to O while minimizing the total travel distance.

#### Loading packages

In [3]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

#### Creating Data
Locations: O, A, B, and C. Office O is the starting and ending point.
The distances are the same in both directions. The model uses directed arcs to represent travel between locations.

In [ ]:
O, A, B, C = "O", "A", "B", "C"
DEPOT = O

distance_data = pd.DataFrame(columns=["From", "To", "Distance"])

distance_data["From"] = [O, O, O, O, A, A, A, A, B, B, B, B, C, C, C, C]

distance_data["To"] = [O, A, B, C, O, A, B, C, O, A, B, C, O, A, B, C]

distance_data["Distance"] = [0, 7, 12, 9, 7, 0, 6, 8, 12, 6, 0, 5, 9, 8, 5, 0]

distance_data

,From,To,Distance
0,O,O,0
1,O,A,7
2,O,B,12
3,O,C,9
4,A,O,7
5,A,A,0
6,A,B,6
7,A,C,8
8,B,O,12
9,B,A,6


#### Define Model

In [5]:
# create empty model
m = AbstractModel()

**Sets and parameters**

- $V = \{O, A, B, C\}$: set of campus locations
- $A = \{(i,j) \in V \times V : i \ne j\}$: set of arcs (no self-loops)
- $d_{ij}$: distance from location $i$ to location $j$ in miles
- $n = |V| = 4$: total number of locations

In [6]:
# define sets
m.V = Set()
m.A = Set(
    within=m.V * m.V, initialize=lambda m: [(i, j) for i in m.V for j in m.V if i != j]
)

# define parameters
m.d = Param(m.V, m.V)
m.n = Param(initialize=lambda m: len(m.V))

**Decision variables**

In [7]:
m.x = Var(m.A, domain=Binary)

m.u = Var(
    m.V,
    domain=NonNegativeIntegers,
    bounds=lambda m, i: (0, 0) if i == DEPOT else (1, m.n - 1),
)

#### Constraints

**Leave each location exactly once**

In [8]:
def Leave_Once_Rule(m, i):
    return sum(m.x[i, j] for j in m.V if j != i) == 1

m.LeaveOnce = Constraint(m.V, rule=Leave_Once_Rule)

**Enter each location exactly once**

In [9]:
def Enter_Once_Rule(m, j):
    return sum(m.x[i, j] for i in m.V if i != j) == 1

m.EnterOnce = Constraint(m.V, rule=Enter_Once_Rule)

**Subtour elimination (MTZ)**

The MTZ constraints prevent separate loops by requiring the visit order to increase when the courier travels directly between two non-depot locations.

In [10]:
def MTZ_Rule(m, i, j):
    if i == DEPOT or j == DEPOT:
        return Constraint.Skip

    return m.u[i] - m.u[j] + (m.n - 1) * m.x[i, j] <= m.n - 2

m.MTZ = Constraint(m.A, rule=MTZ_Rule)

**Objective: Minimize total distance**

In [11]:
def Min_Distance_Rule(m):
    return sum(m.d[i, j] * m.x[i, j] for (i, j) in m.A)

m.Min_Distance = Objective(rule=Min_Distance_Rule, sense=minimize)

#### Create Instance

In [13]:
instanceData = {
    None: {
        "V": {None: list(distance_data["From"].unique())},
        "d": distance_data.set_index(["From", "To"])["Distance"].to_dict(),
    }
}

instance = m.create_instance(instanceData)

#### Solve

In [14]:
# solve instance
solver = SolverFactory("glpk")
solution = solver.solve(instance)

print("Solver Status:")
print(solution["Solver"])

print("Optimum Objective Function Value:")
print(value(instance.Min_Distance))

Solver Status:

- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 5
      Number of created subproblems: 5
  Error rc: 0
  Time: 0.007307767868041992

Optimum Objective Function Value:
27.0


#### Print solution

In [16]:
# create output data: follow the arcs from the depot to get the tour in order
next_city = {i: j for (i, j) in instance.A if value(instance.x[i, j]) > 0.5}

tour = [DEPOT]
while next_city[tour[-1]] != DEPOT:
    tour.append(next_city[tour[-1]])
tour.append(DEPOT)

output_route = pd.DataFrame(
    [
        {"Stop": k + 1, "From": i, "To": j, "Distance": value(instance.d[i, j])}
        for k, (i, j) in enumerate(zip(tour[:-1], tour[1:]))
    ]
)

print("===== OPTIMAL TOUR =====")
print(" → ".join(tour))
print(f"Total Distance: {value(instance.Min_Distance):.0f}")

print("\nSelected decision variables:")
for i, j in instance.A:
    if value(instance.x[i, j]) > 0.5:
        print(f"x[{i},{j}] = {value(instance.x[i,j]):.0f}")

# print ordering variables, excluding the depot
print("\nOrdering variables:")
for i in instance.V:
    if i != DEPOT:
        print(f"u[{i}] = {value(instance.u[i]):.0f}")

output_route

===== OPTIMAL TOUR =====
O → A → B → C → O
Total Distance: 27

Selected decision variables:
x[O,A] = 1
x[A,B] = 1
x[B,C] = 1
x[C,O] = 1

Ordering variables:
u[A] = 1
u[B] = 2
u[C] = 3


,Stop,From,To,Distance
0,1,O,A,7
1,2,A,B,6
2,3,B,C,5
3,4,C,O,9


#### Explanation of Results

The optimal route is O → A → B → C → O, with a total travel distance of 27 miles. The reverse route is also optimal since the distances are same.

The route visits each campus location exactly once and returns to the dispatch office. The arrival and departure constraints are satisfied, and the MTZ constraints prevent disconnected subtours.

The solver reports an optimal solution - it is not just a local optimum - that is 27 miles is the minimum possible distance.